# 📘 Notebook 02 — Préparation et Nettoyage des Données

## Approche scientifique et méthodologique
Ce notebook suit une démarche rigoureuse inspirée des protocoles de traitement des données en hydrologie, chimie environnementale et data science appliquée.

Il comporte les sections suivantes :

- 1 Chargement des données brutes

- 2 Analyse scientifique des valeurs manquantes

- 3 Imputation statistique des valeurs manquantes

- 4 Détection et traitement des valeurs extrêmes (outliers)

- 5 Normalisation / Standardisation des variables

- 6 Export du dataset nettoyé pour les phases suivantes

# 🧪 Cellule 1 — Chargement des données brutes

In [1]:
import pandas as pd

# Chargement des données brutes
df = pd.read_csv("../data/raw/water_potability.csv")

# Affichage des premières lignes
df.head()


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0


# 🧪 Cellule 2 — Analyse des valeurs manquantes

In [2]:
import pandas as pd

# Quantification des valeurs manquantes
missing_counts = df.isna().sum()
missing_percent = (missing_counts / len(df)) * 100

missing_report = pd.DataFrame({
    "Missing_Count": missing_counts,
    "Missing_Percent": missing_percent
})

missing_report


,Missing_Count,Missing_Percent
ph,491,14.987790
Hardness,0,0.000000
Solids,0,0.000000
Chloramines,0,0.000000
Sulfate,781,23.840049
Conductivity,0,0.000000
Organic_carbon,0,0.000000
Trihalomethanes,162,4.945055
Turbidity,0,0.000000
Potability,0,0.000000


# 🧪 Cellule 3 — Classification des variables selon leur criticité

In [3]:
import pandas as pd

# Classification scientifique des variables selon leur criticité
criticality = {
    "ph": "Critique (paramètre OMS : 6.5–8.5, indicateur d’équilibre acido-basique)",
    "Hardness": "Modérée (influence le confort, non critique OMS)",
    "Solids": "Critique (TDS, seuil OMS 500–1000 mg/L)",
    "Chloramines": "Critique (désinfectant réglementé EPA, seuil 4 mg/L)",
    "Sulfate": "Modérée à critique (variabilité naturelle élevée, seuils OMS)",
    "Conductivity": "Modérée (indicateur indirect de minéralisation)",
    "Organic_carbon": "Critique (TOC, indicateur de matière organique, seuil EPA)",
    "Trihalomethanes": "Critique (sous-produits de chloration, seuil EPA 80 µg/L)",
    "Turbidity": "Critique (OMS : < 5 NTU, indicateur de contamination)",
    "Potability": "Variable cible"
}

criticality_report = pd.DataFrame.from_dict(criticality, orient="index", columns=["Criticality"])
criticality_report


,Criticality
ph,"Critique (paramètre OMS : 6.5–8.5, indicateur ..."
Hardness,"Modérée (influence le confort, non critique OMS)"
Solids,"Critique (TDS, seuil OMS 500–1000 mg/L)"
Chloramines,"Critique (désinfectant réglementé EPA, seuil 4..."
Sulfate,Modérée à critique (variabilité naturelle élev...
Conductivity,Modérée (indicateur indirect de minéralisation)
Organic_carbon,"Critique (TOC, indicateur de matière organique..."
Trihalomethanes,"Critique (sous-produits de chloration, seuil E..."
Turbidity,"Critique (OMS : < 5 NTU, indicateur de contami..."
Potability,Variable cible


# 🧪 Cellule 4 — Justification de la méthode d’imputation

In [4]:
import pandas as pd

# Rapport scientifique sur les méthodes d'imputation
report = {
    "Méthode": [
        "Moyenne",
        "Médiane",
        "KNN Imputer",
        "Imputation multiple (MICE)"
    ],
    "Avantages": [
        "Simple, rapide, adaptée aux distributions normales",
        "Robuste aux outliers, adaptée aux distributions asymétriques",
        "Capture les relations entre variables, imputation contextuelle",
        "Méthode statistique avancée, réduit le biais d'imputation"
    ],
    "Inconvénients": [
        "Très sensible aux outliers, biais important dans ce dataset",
        "Ne capture pas les relations entre variables",
        "Coût computationnel élevé, nécessite normalisation préalable",
        "Complexité élevée, nécessite plusieurs modèles, surdimensionné ici"
    ],
    "Pertinence_pour_ce_dataset": [
        "Faible (présence d'outliers et distributions non normales)",
        "Élevée (distributions hétérogènes, valeurs extrêmes)",
        "Modérée (possible mais non nécessaire pour ce taux de NaN)",
        "Faible (dataset trop petit, complexité injustifiée)"
    ]
}

imputation_report = pd.DataFrame(report)
imputation_report


,Méthode,Avantages,Inconvénients,Pertinence_pour_ce_dataset
0,Moyenne,"Simple, rapide, adaptée aux distributions norm...","Très sensible aux outliers, biais important da...",Faible (présence d'outliers et distributions n...
1,Médiane,"Robuste aux outliers, adaptée aux distribution...",Ne capture pas les relations entre variables,"Élevée (distributions hétérogènes, valeurs ext..."
2,KNN Imputer,"Capture les relations entre variables, imputat...","Coût computationnel élevé, nécessite normalisa...",Modérée (possible mais non nécessaire pour ce ...
3,Imputation multiple (MICE),"Méthode statistique avancée, réduit le biais d...","Complexité élevée, nécessite plusieurs modèles...","Faible (dataset trop petit, complexité injusti..."


# 🧪 Cellule 5 — Imputation des valeurs manquantes (méthode : médiane)

La médiane est choisie comme méthode d’imputation car :

- elle est robuste aux outliers, très présents dans les données environnementales ;

- elle ne suppose aucune normalité de distribution ;

- elle est recommandée dans les protocoles OMS/EPA lorsque les mesures présentent une variabilité naturelle élevée ;

- elle évite les biais introduits par la moyenne dans les distributions asymétriques (TDS, Conductivity, Organic Carbon).

Nous allons donc imputer les NaN des colonnes :

- ph

- Sulfate

- Trihalomethanes

In [5]:
import pandas as pd

# Copie du dataset pour éviter toute modification accidentelle
df_clean = df.copy()

# Liste des colonnes à imputer
cols_to_impute = ["ph", "Sulfate", "Trihalomethanes"]

# Imputation par la médiane
for col in cols_to_impute:
    median_value = df_clean[col].median()
    df_clean[col].fillna(median_value, inplace=True)

# Vérification post-imputation
df_clean.isna().sum()


C:\Users\Mazar\AppData\Local\Temp\ipykernel_5568\1243259108.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean[col].fillna(median_value, inplace=True)
C:\Users\Mazar\AppData\Local\Temp\ipykernel_5568\1243259108.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

ph                 0
Hardness           0
Solids             0
Chloramines        0
Sulfate            0
Conductivity       0
Organic_carbon     0
Trihalomethanes    0
Turbidity          0
Potability         0
dtype: int64

# 🧪 Cellule 6 — Détection des outliers (méthode IQR)

Identifier les valeurs extrêmes selon une méthode statistique standard en hydrologie, chimie environnementale et data science :

- IQR (Interquartile Range)

- Détection des points situés hors de l’intervalle :
    - [Q1-1.5xIQR, Q3+1.5IQR]

## Pourquoi IQR ?
Scientifiquement, IQR est recommandé pour les données environnementales car :

- il ne suppose aucune normalité de distribution ;

- il est robuste aux valeurs extrêmes ;

- il est utilisé dans les protocoles OMS/EPA pour les mesures physico‑chimiques ;

- il permet d’identifier les anomalies de mesure ou les conditions environnementales atypiques.

Nous allons produire un rapport d’outliers par variable, sans les supprimer pour l’instant

In [6]:
import pandas as pd

# Détection des outliers par la méthode IQR
outlier_report = {}

for col in df_clean.columns:
    if col != "Potability":  # On ne traite pas la variable cible
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)]
        outlier_report[col] = len(outliers)

# Rapport sous forme de DataFrame
outlier_df = pd.DataFrame.from_dict(outlier_report, orient="index", columns=["Outlier_Count"])
outlier_df


,Outlier_Count
ph,142
Hardness,83
Solids,47
Chloramines,61
Sulfate,264
Conductivity,11
Organic_carbon,25
Trihalomethanes,54
Turbidity,19


# 🧪 Cellule 7 — Décision : conserver ou supprimer les outliers

Déterminer, pour chaque variable, si les outliers :

- représentent des conditions environnementales extrêmes mais réelles,

- ou des erreurs de mesure / anomalies instrumentales,

- et donc s’ils doivent être conservés ou supprimés.

Cette étape est essentielle pour éviter :

- de détruire des informations utiles (ex : eau très minéralisée, turbidité élevée),

- ou au contraire de laisser des valeurs aberrantes qui dégradent la modélisation.

# 📊 Interprétation des outliers après recherche

| Variable           | Outliers | Interprétation scientifique                                                                                   |
|--------------------|----------|----------------------------------------------------------------------------------------------------------------|
| ph                 | 142      | Valeurs extrêmes possibles (eaux acides/basiques), mais > 12 ou < 3 sont souvent des erreurs → à filtrer partiellement |
| Hardness           | 83       | Dureté très variable selon les régions → à conserver                                                          |
| Solids (TDS)       | 47       | Très normal : eaux très minéralisées → à conserver                                                             |
| Chloramines        | 61       | Valeurs > 10 mg/L sont physiquement improbables → à filtrer                                                   |
| Sulfate            | 264      | Très variable naturellement → à conserver sauf valeurs > 500 mg/L                                             |
| Conductivity       | 11       | Normal, reflète la minéralisation → à conserver                                                                |
| Organic_carbon     | 25       | Valeurs extrêmes possibles (pollution organique) → à conserver                                                |
| Trihalomethanes    | 54       | Valeurs > 120 µg/L sont improbables → à filtrer                                                                |
| Turbidity          | 19       | Valeurs > 10 NTU sont plausibles → à conserver                                                                 |


# 🧪 Décision

- 🔵 Conserver les outliers (réels, environnementaux)
Hardness

    - Solids

    - Conductivity

    - Organic_carbon

    - Turbidity

    - Sulfate (sauf valeurs > 500 mg/L)

- 🔴 Filtrer les outliers (probablement des erreurs de mesure)
    - ph

    - Chloramines

    - Trihalomethanes

    - Sulfate > 500 mg/L

# 🧪 Cellule 7 — Filtrage des outliers

In [7]:
import pandas as pd

df_filtered = df_clean.copy()

# Filtrage scientifique des outliers
df_filtered = df_filtered[
    (df_filtered["ph"] >= 3) & (df_filtered["ph"] <= 12) &
    (df_filtered["Chloramines"] <= 10) &
    (df_filtered["Trihalomethanes"] <= 120) &
    (df_filtered["Sulfate"] <= 500)
]

# Vérification du nombre de lignes après filtrage
df_filtered.shape


(3147, 10)

# 🧪 Cellule 8 — Normalisation des variables (StandardScaler)

## Normaliser les variables afin de :

- homogénéiser les échelles (TDS, Conductivity, Organic Carbon ont des ordres de grandeur très différents)

- stabiliser les modèles non linéaires (RandomForest, GradientBoosting)

- éviter que certaines variables dominent artificiellement l’apprentissage

- respecter les bonnes pratiques OMS/EPA pour l’analyse multivariée des paramètres physico‑chimiques

## Pourquoi StandardScaler ?
Scientifiquement, StandardScaler est recommandé car :

- il centre les données (moyenne = 0)

- il réduit la variance (écart-type = 1)

- il est adapté aux modèles non linéaires

- il conserve les relations entre variables

- il ne déforme pas les distributions comme MinMaxScaler peut le faire

In [8]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Copie du dataset filtré
df_scaled = df_filtered.copy()

# Sélection des colonnes à normaliser (toutes sauf la cible)
features = df_scaled.drop("Potability", axis=1).columns

# Initialisation du scaler
scaler = StandardScaler()

# Normalisation
df_scaled[features] = scaler.fit_transform(df_scaled[features])

# Vérification statistique post-normalisation
df_scaled[features].describe()


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity
count,3.147000e+03,3.147000e+03,3.147000e+03,3.147000e+03,3.147000e+03,3.147000e+03,3.147000e+03,3.147000e+03,3.147000e+03
mean,-1.298259e-16,6.841260e-16,6.536451e-16,5.847810e-16,-1.025060e-15,-1.467597e-16,1.332127e-16,1.512754e-16,-2.190106e-16
std,1.000159e+00,1.000159e+00,1.000159e+00,1.000159e+00,1.000159e+00,1.000159e+00,1.000159e+00,1.000159e+00,1.000159e+00
min,-2.862506e+00,-3.802459e+00,-2.494089e+00,-4.587638e+00,-5.794777e+00,-3.016282e+00,-3.668687e+00,-4.163729e+00,-3.225740e+00
25%,-5.663353e-01,-5.969770e-01,-7.266535e-01,-6.244582e-01,-4.578383e-01,-7.455094e-01,-6.644295e-01,-6.149121e-01,-6.752174e-01
50%,-4.255169e-02,1.913468e-02,-1.188081e-01,4.998208e-02,-2.306686e-02,-5.717144e-02,-2.000453e-02,1.242833e-02,-1.772741e-02
75%,5.538890e-01,6.187059e-01,6.037938e-01,6.973954e-01,4.683612e-01,6.889452e-01,6.851621e-01,6.510814e-01,6.818976e-01
max,3.448451e+00,3.912770e+00,4.488870e+00,2.068022e+00,4.161530e+00,4.034478e+00,4.235789e+00,3.291692e+00,3.554096e+00


# 🧪 Cellule 9 — Export du dataset nettoyé

Exporter le dataset nettoyé dans un format stable, reproductible, et utilisable pour :

- le feature engineering (Notebook 03)

- la modélisation (Notebook 04)

- l’intégration API (Notebook 05)

In [9]:
# Export du dataset nettoyé
df_scaled.to_csv("../data/processed/water_clean.csv", index=False)

# Vérification
df_scaled.head()


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,-0.042552,0.258592,-0.147141,0.206758,0.979347,1.703775,-1.191411,1.303506,-1.286090,0
1,-2.422454,-2.073841,-0.394932,-0.252081,-0.023067,2.056113,0.262355,-0.640030,0.684819,0
2,0.718843,0.856502,-0.248238,1.570008,-0.023067,-0.092670,0.773761,-0.000401,-1.167134,0
3,0.874825,0.551676,-0.006453,0.730565,0.650413,-0.774983,1.248601,2.149755,0.849046,0
4,1.430590,-0.476639,-0.469578,-0.313248,-0.671805,-0.341670,-0.834499,-2.182281,0.139278,0


# 📄 Conclusion

Le protocole de préparation des données a permis d’obtenir un jeu de données propre, cohérent et scientifiquement exploitable pour les phases de modélisation. Les étapes réalisées incluent :

## 1. Imputation statistique des valeurs manquantes
Les variables ph, Sulfate et Trihalomethanes ont été imputées par la médiane, méthode robuste adaptée aux distributions environnementales hétérogènes.

## 2. Filtrage scientifique des valeurs extrêmes
Un filtrage sélectif basé sur les seuils OMS/EPA a été appliqué aux variables susceptibles de contenir des erreurs instrumentales (pH, chloramines, trihalométhanes, sulfate > 500 mg/L).
Les autres outliers ont été conservés car ils reflètent des conditions environnementales réelles.

## 3. Normalisation des variables
Les paramètres physico‑chimiques ont été normalisés via StandardScaler, garantissant une homogénéité des échelles et une stabilité des modèles non linéaires.

## 4. Export du dataset nettoyé
Le dataset final, composé de 3147 observations, a été exporté dans data/processed/water_clean.csv pour les étapes suivantes.

## Le dataset est maintenant prêt pour :
- Notebook 03 : Feature engineering 

- Notebook 04 : Modélisation non linéaire (RandomForest, GradientBoosting)

- Notebook 05 : Intégration API / Monitoring